In [1]:
import kglab
import pickle
import rdflib 
import re
import torch
import json
import gc
import os
import logging
import random

import networkx as nx
import pandas as pd
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm
from collections import Counter, defaultdict
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer, losses, models, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.datasets import DenoisingAutoEncoderDataset
from torch.utils.data import DataLoader
from datasets import Dataset, IterableDataset
from transformers import BertLMHeadModel

tqdm.pandas()

C:\Users\roans\anaconda3.0\envs\transformers\lib\site-packages\kglab\util.py:35: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore  # pylint: disable=E0401


In [2]:
triples = pd.read_excel("../outputs/clean_outputs/filtered_triples.xlsx").drop("Unnamed: 0", axis=1)
triples.head()

,id,company,job title,text,triples_qwen_structured,triples_qwen_semi-structured,triples_qwen_unstructured,triples_gemma_structured,triples_gemma_semi-structured,triples_gemma_unstructured,triples_llama_structured,triples_llama_semi-structured,triples_llama_unstructured
0,1527392,Jobindex,"IT-administrator – få indflydelse på et setup,...",Vil du ind i en virksomhed i rivende udvikling...,"[('IT-administrator', 'REQUIRES_SKILL', 'setup...","[('IT-administrator', 'REQUIRES_SKILL', 'netwo...","[('IT-administrator', 'requires', 'technical_s...","[('IT-administrator', 'INVOLVES_TASK', 'Mainta...","[('IT-administrator', 'REQUIRES_SKILL', 'Syste...",[('IT-administrator – få indflydelse på et set...,"[('IT-administrator', 'REQUIRES_SKILL', 'setup...","[('IT-administrator', 'REQUIRES_SKILL', 'Docke...","[('IT-administrator', 'har', 'få indflydelse p..."
1,1527395,Jobindex,"IT-administrator – få indflydelse på et setup,...","IT-administrator – få indflydelse på et setup,...","[('IT-administrator', 'REQUIRES_SKILL', 'setup...","[('IT-administrator', 'REQUIRES_SKILL', 'netwo...","[('IT-administrator', 'requires', 'technical_s...","[('IT-administrator', 'REQUIRES_SKILL', 'IT Ad...","[('IT-administrator', 'REQUIRES_SKILL', 'Syste...",[('IT-administrator – få indflydelse på et set...,"[('IT-administrator', 'REQUIRES_SKILL', 'Docke...",[('IT-administrator – få indflydelse på et set...,"[('IT-administrator', 'har', 'få indflydelse p..."
2,1527397,Aqua d'Or Mineral Water A/S,SQE Manager,For jobsøgere For arbejdsgivere Aqua d'Or Mi...,"[('SQE Manager', 'REQUIRES_SKILL', 'Root Cause...","[('SQE Manager', 'REQUIRES_SKILL', 'Quality Ma...","[('SQE Manager', 'requires', 'quality assuranc...","[('SQE Manager', 'REQUIRES_SKILL', 'Quality As...","[('SQE Manager', 'REQUIRES_SKILL', 'Quality As...","[('SQE Manager', 'title', 'SQE Manager'), ('SQ...","[('SQE Manager', 'REQUIRES_QUALITY', 'detail-o...","[('SQE Manager', 'REQUIRES_SKILL', 'DevOps Eng...","[('SQE Manager', 'requires', 'English language..."
3,1527417,Klimabrands,Kundeservice / teknisk support,For jobsøgere For arbejdsgivere mailto:job@k...,"[('Kundeservice / teknisk support', 'REQUIRES_...","[('Kundeservice / teknisk support', 'REQUIRES_...","[('Kundeservice / teknisk support', 'requires'...","[('Kundeservice / teknisk support', 'REQUIRES_...","[('Kundeservice / teknisk support', 'REQUIRES_...","[('Kundeservice / teknisk support', 'title', '...","[('Kundeservice', 'REQUIRES_QUALITY', 'detail-...","[('Kundeservice', 'REQUIRES_SKILL', 'Teknisk s...","[('Kundeservice', 'har', 'en teknisk support s..."
4,1527440,Scan Studio ApS,Retail designer med teknikken på plads,For jobsøgere For arbejdsgivere Scan Studio ...,"[('Retail Designer', 'REQUIRES_SKILL', 'Techni...","[('Retail Designer', 'REQUIRES_SKILL', 'Design...","[('Retail Designer', 'requires', 'technical sk...","[('Retail designer', 'REQUIRES_SKILL', 'teknik...","[('Retail designer', 'REQUIRES_SKILL', 'teknik...","[('Retail designer med teknikken på plads', 'h...","[('Retail designer med teknikken på plads', 'R...","[('Dansk designer', 'REQUIRES_QUALITY', 'SKILL...",[('Retail designer requires technisk kompetenc...


In [3]:
def extract_triples(input_string):
    """
    Extracts only 3-element tuples, accepting single or double quotes.
    """

    input_string = str(input_string)
    processed_string = input_string.replace('*', '"')

    # Quoting Group (Q): This group (r'["\']') matches either a double quote OR a single quote.
    Q = r'["\']' 
        
    # Flexible Pattern (using f-string for clarity and Q definition):
    pattern = rf"""
        \(              # Match the literal opening parenthesis (
        ({Q}.*?{Q})     # Group 1: Capture the first quoted string
        ,\s* # Match comma, optional whitespace
        ({Q}.*?{Q})     # Group 2: Capture the second quoted string
        ,\s* # Match comma, optional whitespace
        ({Q}.*?{Q})     # Group 3: Capture the third quoted string
        \)              # Match the literal closing parenthesis )
    """
    
    # Use re.findall with re.VERBOSE for multiline pattern and re.DOTALL to match across newlines
    matches = re.findall(pattern, processed_string, re.VERBOSE | re.DOTALL)
    
    extracted_data = []
    for str1_quoted, str2_quoted, str3_quoted in matches:
        # Remove the surrounding quotes from each captured string
        # using the replace method, which handles both ' and "
        str1 = str1_quoted.strip().replace('"', '').replace("'", '')
        str2 = str2_quoted.strip().replace('"', '').replace("'", '')
        str3 = str3_quoted.strip().replace('"', '').replace("'", '')
        extracted_data.append((str1, str2, str3))
        
    return extracted_data

for model in ["qwen", "gemma", "llama"]:
    for prompt in ["structured", "semi-structured", "unstructured"]:
        triples[f"triples_{model}_{prompt}"] = triples[f"triples_{model}_{prompt}"].apply(extract_triples)

In [4]:
def clean_entity(entity):
    return entity.replace("_", " ").lower()

def extract_entities(triples):
    return list(np.array([[clean_entity(i[0]), clean_entity(i[2])] for i in triples]).flatten().tolist())

all_entities = []

for model in ["qwen", "gemma", "llama"]:
    for prompt in ["structured", "semi-structured", "unstructured"]:
        all_entities.extend(triples[f"triples_{model}_{prompt}"].progress_apply(extract_entities).values)

entity_list = list(set([
    entity
    for entities in all_entities
    for entity in entities
]))

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

In [5]:
len(entity_list)

535029

In [6]:
with open("../../dataset/final_dataset/anon_cvs.json", 'r', encoding="utf-8") as f:
    data = json.load(f)

In [7]:
cv_entities = []

for sample in data:
    for l in ["job_history", "jobtitles", "keywords"]:
        if type(sample[l]) == list:
            cv_entities.extend(sample[l])

cv_entities = set([clean_entity(i) for i in cv_entities if type(i) == str])  
entity_list.extend(list(cv_entities))

In [11]:
# 1. Setup Device and Data
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

unique_entities = list(set(entity_list))
random.seed(42)
subset_entities = random.sample(unique_entities, min(100000, len(unique_entities)))

# The loss function expects 2 inputs. 
# We provide the sentence twice: once for the noisy input, once for the target.
train_dataset = Dataset.from_dict({
    "sentence1": subset_entities, 
    "sentence2": subset_entities
})

# 2. Setup Model
word_embedding_model = models.Transformer('jjzha/dajobbert-base-uncased')
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode='mean')
model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

# 3. Manual Weight Tying (Fixes the AttributeError in transformers 4.40+)
decoder = BertLMHeadModel.from_pretrained('jjzha/dajobbert-base-uncased')
decoder.bert.encoder = model.transformers_model.encoder
decoder.bert.embeddings = model.transformers_model.embeddings

# Initialize Loss - tie_encoder_decoder=False because we just did it manually above
train_loss = losses.DenoisingAutoEncoderLoss(
    model, 
    decoder_name_or_path='jjzha/dajobbert-base-uncased', 
    tie_encoder_decoder=False 
)
train_loss.decoder = decoder.to(device)

# 4. Define Training Arguments
args = SentenceTransformerTrainingArguments(
    output_dir="dajobbert-checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=32, # Good balance for 12GB VRAM
    learning_rate=3e-5,
    warmup_ratio=0.1,
    fp16=True, # Uses your 4070 Ti Tensor Cores
    save_steps=1000,
    logging_steps=100,
    report_to="none"
)

# 5. Initialize Trainer
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
)

# 6. Train and Save
trainer.train()
model.save('dajobbert-kg-specialized')
print("Training complete! Model saved as 'dajobbert-kg-specialized'")

Using device: cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: jjzha/dajobbert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertLMHeadModel LOAD REPORT from: jjzha/dajobbert-base-uncased
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete! Model saved as 'dajobbert-kg-specialized'


In [8]:
class TensorEncoder(json.JSONEncoder):
    def default(self, obj):
        if torch.is_tensor(obj):
            return obj.tolist()
        return super().default(obj)

In [9]:
if "entity_embeddings.json" in os.listdir():
    with open("entity_embeddings.json") as f:
        entity_embs = json.load(f)
else:
    device = ("cuda:0" if torch.cuda.is_available() else "cpu")
    print(device)
    model = SentenceTransformer('dajobbert-kg-specialized').to(device)
    
    entity_embs = {}
    
    for entity in tqdm(entity_list):
        if type(entity) == float and np.isnan(entity):
            continue
        
        entity_embs[entity] = model.encode(entity.replace("_", " "), convert_to_tensor=True)

    with open("entity_embeddings.json", "w") as fp:
        json.dump(entity_embs, fp, cls=TensorEncoder)

entity_list = list(entity_embs.keys())

del model
torch.cuda.empty_cache() 
torch.cuda.ipc_collect()
gc.collect()

30

In [18]:
ordered_keys = list(entity_embs.keys())
entity_list = ordered_keys
ordered_embeddings = [torch.Tensor(entity_embs[key]) for key in ordered_keys]

embeddings = torch.stack(ordered_embeddings).to("cpu")
embeddings_array = F.normalize(embeddings, p=2, dim=1).to(torch.float32).numpy()
# similarity_matrix = cosine_similarity(embeddings_array.numpy(), embeddings_array.numpy())

In [19]:
embeddings_array.shape

(662278, 768)

In [22]:
def resolve_entities(embeddings, threshold=0.95, top_k=5, batch_size=10000):
    # 1. Standard Conversion & Normalization
    if hasattr(embeddings, 'detach'): embeddings = embeddings.detach().cpu().numpy()
    embeddings = embeddings.astype('float32')
    
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings /= (norms + 1e-10) # Avoid division by zero

    d = embeddings.shape[1]
    n = embeddings.shape[0]

    # 2. Build Index
    index = faiss.IndexFlatIP(d)
    index.add(embeddings)

    # 3. Batched Search with Progress Bar
    all_distances = []
    all_indices = []
    
    print(f"Searching {n} entities...")
    for i in tqdm(range(0, n, batch_size)):
        batch = embeddings[i : i + batch_size]
        distances, indices = index.search(batch, top_k)
        all_distances.append(distances)
        all_indices.append(indices)

    # Concatenate results back together
    distances = np.vstack(all_distances)
    indices = np.vstack(all_indices)

    # 4. Filter Results
    duplicates = []
    for i in range(n):
        for j in range(1, top_k):
            if distances[i, j] >= threshold:
                neighbor_idx = indices[i, j]
                if i < neighbor_idx:
                    duplicates.append((i, neighbor_idx, distances[i, j]))
                    
    return duplicates

matches = resolve_entities(embeddings_array, threshold=0.98)
print(f"Found {len(matches)} potential duplicate pairs.")

Searching 662278 entities...


  0%|          | 0/67 [00:00<?, ?it/s]

Found 69455 potential duplicate pairs.


In [30]:
import networkx as nx
import community as community_louvain # from python-louvain

def cluster_with_louvain(num_entities, duplicate_pairs):
    """
    Uses the Louvain algorithm to partition entities into dense communities.
    
    Args:
        num_entities: Total original entity count (600k)
        duplicate_pairs: List of (idx1, idx2, score)
    """
    G = nx.Graph()
    G.add_nodes_from(range(num_entities))
    
    # We add weights. Louvain uses these to determine tie-strength.
    # Higher score = stronger bond.
    for u, v, score in duplicate_pairs:
        G.add_edge(u, v, weight=score)
    
    print("Running Louvain community detection...")
    # resolution=1.0 is standard. 
    # Increase it (>1.0) for more, smaller clusters.
    # Decrease it (<1.0) for fewer, larger clusters.
    partition = community_louvain.best_partition(G, weight='weight', resolution=1.0)
    
    # Louvain returns {node_id: cluster_id}. 
    # We'll also group them into a list of lists for consistency with your previous flow.
    clusters = {}
    for node, cluster_id in partition.items():
        clusters.setdefault(cluster_id, []).append(node)
    
    cluster_list = list(clusters.values())
    
    return partition, cluster_list

cluster_map, clusters = cluster_with_louvain(len(embeddings), matches)

Running Louvain community detection...


In [31]:
canonical_id_map = {}
for cluster in clusters:
    leader_idx = min(cluster)  # The representative ID for this group
    for member_idx in cluster:
        canonical_id_map[member_idx] = leader_idx

In [36]:
def create_canonical_string_map(entity_list, clusters):
    """
    Creates a mapping from every original string to its cluster's canonical string.
    
    Args:
        entity_list: List of 600k entity names (strings)
        clusters: List of lists containing indices (from our clustering step)
    """
    string_map = {}
    
    for cluster in clusters:
        # 1. Pick a leader index (e.g., the smallest index in the cluster)
        leader_idx = min(cluster)
        canonical_name = entity_list[leader_idx]
        
        # 2. Map every entity in this cluster to that canonical name
        for member_idx in cluster:
            original_name = entity_list[member_idx]
            string_map[original_name] = canonical_name
            
    return string_map

# Usage:
canonical_names = create_canonical_string_map(entity_list, clusters)

In [32]:
from collections import Counter

def check_cluster_stats(clusters):
    sizes = [len(c) for c in clusters]
    size_counts = Counter(sizes)
    
    print("--- Cluster Statistics ---")
    print(f"Total Clusters: {len(clusters)}")
    print(f"Singletons (Unique Entities): {size_counts[1]}")
    print(f"Resolved (Clusters with >1 member): {len(clusters) - size_counts[1]}")
    
    print("\n--- Top 5 Largest Clusters ---")
    sorted_sizes = sorted(sizes, reverse=True)
    for i, s in enumerate(sorted_sizes[:5]):
        print(f"Cluster {i+1}: {s} entities merged")
        
    if sorted_sizes[0] > 100:
        print("\nWARNING: Large cluster detected. You might need a higher threshold or community detection.")

# Run the check
check_cluster_stats(clusters)

--- Cluster Statistics ---
Total Clusters: 610440
Singletons (Unique Entities): 587520
Resolved (Clusters with >1 member): 22920

--- Top 5 Largest Clusters ---
Cluster 1: 207 entities merged
Cluster 2: 193 entities merged
Cluster 3: 151 entities merged
Cluster 4: 147 entities merged
Cluster 5: 141 entities merged



In [40]:
def map_list_of_triples_to_canonical(list_of_triples, canonical_map):
    """
    Iterates over a list of triples and updates the first (subject) 
    and third (object) elements of each triple using the canonical map.

    :param list_of_triples: A list where each item is a triple (a, b, c).
    :param canonical_map: The dictionary mapping original strings to canonical strings.
    :return: A new list of triples with canonicalized entities.
    """
    canonicalized_list = []

    for triple in list_of_triples:
        # Assuming the triple is iterable (e.g., a list or tuple)
        if len(triple) != 3:
            # Handle malformed triples if necessary, or skip them
            canonicalized_list.append(triple)
            continue
            
        a, b, c = triple

        # Check and replace 'a' (subject)
        # .get(key, default) returns the original 'a' if not found
        new_a = canonical_map.get(a, a) 

        # Check and replace 'c' (object)
        new_c = canonical_map.get(c, c) 

        # Append the new, canonicalized triple
        canonicalized_list.append((new_a, b, new_c))
        
    return canonicalized_list

In [41]:
for model in ["qwen", "gemma", "llama"]:
    for prompt in ["structured", "semi-structured", "unstructured"]:
        triples[f"triples_{model}_{prompt}"] = triples[f"triples_{model}_{prompt}"].apply(
            lambda list_of_triples: map_list_of_triples_to_canonical(list_of_triples, canonical_names)
        )

In [43]:
triples.to_excel("../outputs/clean_outputs/triples_coalesced.xlsx")

In [44]:
def map_json_entities_to_canonical(json_list, canonical_map, keys_to_map=None):
    """
    Updates entities within lists under specified keys in a list of dictionaries (JSON format)
    to their canonical forms.

    :param json_list: The list of dictionaries (your JSON data).
    :param canonical_map: The dictionary mapping original strings to canonical strings.
    :param keys_to_map: The keys whose values (lists of entities) should be mapped.
                        Defaults to ['job_history', 'jobtitles', 'keywords'].
    :return: A new list of dictionaries with canonicalized entity lists.
    """
    if keys_to_map is None:
        keys_to_map = ['job_history', 'jobtitles', 'keywords']
        
    canonicalized_json = []

    for entity_dict in json_list:
        new_dict = {}
        
        # Iterate over all keys in the dictionary
        for key, value in entity_dict.items():
            
            # Check if the key is one we need to map AND if the value is a list
            if key in keys_to_map and isinstance(value, list):
                
                canonicalized_list = []
                # Iterate through every entity in the list value
                for entity in value:
                    # Replace the entity with its canonical form if found in the map
                    # Otherwise, keep the original entity string
                    canonical_entity = canonical_map.get(entity, entity)
                    canonicalized_list.append(canonical_entity)
                
                new_dict[key] = list(set(canonicalized_list))
            
            # If the key is not one to map, or the value is not a list, copy it as is
            else:
                new_dict[key] = value
                
        canonicalized_json.append(new_dict)
        
    return canonicalized_json

In [45]:
coalesced_data = map_json_entities_to_canonical(data, canonical_names)

In [46]:
coalesced_data[:10]

[{'headline': 'Erfaren Senior IT SystemKonsulent',
  'educationlevel': 8.0,
  'jobexperiences': 35.0,
  'mgrexperiences': 2.0,
  'countryid': 4,
  'cvid': 'ebf358ed252344af8d1dd3f4c8516cbf',
  'job_history': ['It-drift og support',
   'Cloud Infrastructure Specialist',
   'IT-systemkonsulent',
   'High end drifts konsulent',
   'IT-administrator'],
  'educations': [{'type': 'education', 'name': 'EDB assistent'}],
  'languages': [{'code': 'en', 'level': 4},
   {'code': 'de', 'level': 2},
   {'code': 'da', 'level': 4}],
  'jobtitles': ['Senior konsulent',
   'Systemkonsulent',
   'IT-ansvarlig',
   'IT-systemkonsulent',
   'Systemadministrator',
   'IT-konsulent',
   'Senior Consultant',
   'Systemansvarlig',
   'IT-administrator'],
  'keywords': ['Exchange server',
   'Driftsoptimering',
   'Microsoft Exchange',
   'Problemknuser',
   'Windows',
   'nutanix',
   'Ledelse',
   'Serious and responsible',
   'IT koordinator',
   'Driftsteknik',
   'Azure',
   'IT konsulent',
   'SDM']},
 {

In [47]:
with open("anon_cvs_coalesced.json", 'w') as json_file:
        # json.dump() writes the dictionary to the file object
        json.dump(coalesced_data, json_file, indent=4)